[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Mach_Learn/TinyML.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# TinyML: Neural Networks on Microcontrollers

> ⚠️ **Draft — requires a microcontroller board (e.g. Raspberry Pi Pico, ~$6) not available at authoring time.** An instructor should run each block before teaching; remove this banner after.

The [Model Compression](../Intro_Mach_Learn/Model_Compression.ipynb) pipeline, driven to its destination: a keyword-spotting-style classifier running on a $6 board with 264 KB of RAM — int8 weights exported to a C array, inference in plain C, and the [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb) budget respected.

## 1. Pre-requisites

[Model Compression](../Intro_Mach_Learn/Model_Compression.ipynb), [Intro to C](../Intro_Programming/Intro_C.ipynb), [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb).

---
### 🕐 Session 1 of 3 — *The Budget & the Model* (~35 min)
**Goal:** fit the arithmetic: 264 KB RAM, no FPU worth using — design the model backwards from the board.
**Feeds into:** Session 2 (export to C).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: The Budget & the Model</b></summary>

**Timing (~35 min).** 10 min reading a datasheet as a constraint set · 12 min the three budgets · 8 min designing backwards · 5 min the verify-on-desktop rule.

**First, the practical warning: this workshop needs hardware and is marked draft.** Nothing here executes in the notebook. **Run every block yourself on an actual board before teaching it** — the toolchain, the SDK version, and the flashing path all drift, and discovering that live in front of a room is unrecoverable. A Raspberry Pi Pico is about $6; buy several, because students will brick one.

**Open by inverting the habit the ML track has built.** Every previous workshop asked "what architecture solves this task?" and treated compute as elastic. **On a microcontroller you design backwards from the datasheet.** The RP2040 gives you 264 KB of RAM, 2 MB of flash, 133 MHz, and **no useful FPU**. Those four numbers are the specification; the model is whatever fits inside them.

**Make the three budgets explicit and separate, because students conflate them.**
- **Flash bounds weights.** 2 MB is generous — a 20k-parameter int8 model is 20 KB, so weights are rarely the binding constraint.
- **RAM bounds activations.** This is usually what kills you: a single $32\times32\times16$ int8 feature map is 16 KB, and you need at least two live at once. 264 KB disappears faster than students expect.
- **Clock × cycles-per-MAC bounds latency.** At 133 MHz with a handful of cycles per int8 MAC, a few hundred thousand MACs is tens of milliseconds.

**Have the room do the arithmetic for the compression workshop's student model before you show any answer.** 1.6k parameters at int8 is 1.6 KB of flash — trivial. The activations are the question. **That calculation is the whole session in miniature**, and it is the habit worth installing: compute the budget before writing the model.

**Emphasise "no FPU" as a design constraint rather than a footnote.** The RP2040 has no hardware floating point, so every float operation is a software routine costing tens of cycles. **Integer arithmetic is not an optimisation here, it is the only viable option** — which is why the [Model Compression](../Intro_Mach_Learn/Model_Compression.ipynb) workshop's int8 quantisation is a prerequisite rather than a nicety. Point back at the [FPGA](../Intro_FPGA/Intro_FPGA.ipynb) fixed-point workshop too; it is the same arithmetic discipline.

**Then the workflow rule, and state it as a rule because it is one.** Train in float on the desktop. Quantise post-hoc. **Verify the int8 model's accuracy on the desktop before any code touches the board.** On-device debugging means no debugger worth the name, no printf without a serial link, and a flash-test cycle measured in minutes. Every bug you can catch on a laptop is a bug you do not catch through a blinking LED.

**Close by naming what makes this workshop the destination rather than another technique.** [Model Compression](../Intro_Mach_Learn/Model_Compression.ipynb) measured quantisation, pruning, and distillation as *ratios*. Here those ratios become the difference between a model that runs and one that does not fit in RAM. **The compression ladder was never about elegance; it was about this board.**
</details>

💡 **Intuition.** On a microcontroller you design *backwards from the datasheet*: RAM bounds activations, flash bounds weights, clock × cycles-per-MAC bounds latency. A spectrogram CNN at int8 with ~20k parameters fits a Pico with room to spare — the [compression workshop's](../Intro_Mach_Learn/Model_Compression.ipynb) student model is already nearly deployable. Train in float, quantize post-hoc, *verify the int8 model's accuracy on the desktop first* — debugging on-device is misery you schedule around.

```python
# desktop side (runnable there — reuse Model_Compression's SpecCNN pipeline):
# 1. train tiny CNN on 32x32 spectrograms   2. per-tensor int8 quantization
# 3. export:
def export_c_array(name, tensor_q, scale):
    vals = ', '.join(str(int(v)) for v in tensor_q.flatten())
    return (f'const int8_t {name}[{tensor_q.numel()}] = {{{vals}}};\n'
            f'const float {name}_scale = {scale}f;\n')
```

---
### 🕐 Session 2 of 3 — *Inference in Plain C* (~40 min)
**Goal:** conv/relu/pool/dense as loops over int8 arrays; accumulate in int32 (the headroom rule).
**Builds on:** Session 1; [Intro to C](../Intro_Programming/Intro_C.ipynb). &nbsp; **Feeds into:** Session 3 (on-device).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Inference in Plain C</b></summary>

**Timing (~40 min).** 8 min why write it yourself · 12 min the headroom rule · 12 min requantisation · 8 min the desktop oracle.

**Justify writing the runtime by hand, because students will ask why not TFLite Micro.** In production you *would* use it. **The point of writing it yourself once is that a convolution stops being a library call and becomes six nested loops you can count.** After this session a student can look at a layer and estimate its cycle count, its RAM footprint, and its worst-case latency — which is exactly what deployment requires and what a framework hides.

**Make the headroom rule the centrepiece, since it is the one bug that silently destroys everything.** Inputs are int8 and weights are int8, so each product fits in int16. But a $3\times3$ convolution over $C_{\text{in}}$ channels sums $9 C_{\text{in}}$ of them — at $C_{\text{in}} = 16$ that is 144 terms, each up to $127 \times 127 \approx 16{,}000$, for a worst case near $2.3$ million. **int16 overflows at 32,767.** Hence `int32_t acc`. Ask the room to compute the worst case before revealing it; the number is startling and it makes the rule stick.

**Then explain requantisation, which is where the scales finally earn their keep.** The accumulator is in units of $x_{\text{scale}} \times w_{\text{scale}}$; the next layer expects units of $\text{out}_{\text{scale}}$. So you multiply by $x_s w_s / \text{out}_s$ and clamp back to int8. **Every quantised layer ends with this conversion**, and it is precisely the fixed-point rescaling from [Intro to FPGA](../Intro_FPGA/Intro_FPGA.ipynb) — same arithmetic, different silicon.

**Flag the `float` in the requantisation line as a deliberate simplification with a real cost.** On a chip with no FPU, that division is a software routine costing tens of cycles, executed **once per output element**. Production runtimes replace it with a fixed-point multiply-and-shift: precompute $x_sw_s/\text{out}_s$ as an int32 multiplier plus a right-shift count, and the whole requantisation becomes two integer instructions. **This is the single highest-value optimisation in the file**, and it is a good exercise to assign.

**Point out the clamp and what it silently reveals.** `fmaxf(-128, fminf(127, ...))` is saturation, and after a ReLU the negative branch never fires. But if the clamp at $+127$ fires **often**, your output scale is too small and you are losing information — instrumenting the saturation rate is a genuine debugging technique. **Silent clipping is how quantised models mysteriously lose accuracy.**

**Then make the desktop-oracle rule non-negotiable, because it is the professional practice being taught.** Compile the *same* C on a laptop, run the exported weights against 100 test spectrograms, and demand **bit-identical** agreement with the Python int8 simulation. Not "close" — identical, because integer arithmetic is deterministic and any discrepancy is a bug in indexing, layout, or scale handling. **A mismatch you find on a laptop takes minutes; the same mismatch found through a blinking LED takes a day.**

**If time allows, have the room reason about the loop order.** The code iterates $c_o, i, j$ outermost and $c_i, d_i, d_j$ innermost — which means the weight array is traversed contiguously per output channel while the input is strided. **On a cacheless microcontroller that matters less than on a CPU**, but on any device with a cache the ordering is a several-fold performance difference, and noticing that is the same skill the [Scale_NN](./Scale_NN/Scale_NN.ipynb) memory-bound discussion taught at a different scale.
</details>

```c
// the whole runtime is ~100 lines of this shape (no libraries, no malloc):
void conv2d_int8(const int8_t* x, const int8_t* w, const int32_t* bias,
                 int8_t* out, int H, int W, int Cin, int Cout,
                 float x_scale, float w_scale, float out_scale) {
    for (int co = 0; co < Cout; co++)
      for (int i = 0; i < H; i++)
        for (int j = 0; j < W; j++) {
          int32_t acc = bias[co];               // int32 accumulator: the headroom rule
          for (int ci = 0; ci < Cin; ci++)
            for (int di = -1; di <= 1; di++)
              for (int dj = -1; dj <= 1; dj++) {
                int ii = i+di, jj = j+dj;
                if (ii < 0 || ii >= H || jj < 0 || jj >= W) continue;
                acc += (int32_t)x[(ci*H+ii)*W+jj] * w[((co*Cin+ci)*3+di+1)*3+dj+1];
              }
          float v = acc * x_scale * w_scale / out_scale;  // requantize
          out[(co*H+i)*W+j] = (int8_t)fmaxf(-128, fminf(127, roundf(fmaxf(0, v))));
        }
}
```

**Desktop oracle before flashing:** compile this same C on your laptop, run the exported weights against 100 test spectrograms, and require bit-identical agreement with the Python int8 simulation. Only then touch the board.

---
### 🕐 Session 3 of 3 — *On the Board* (~40 min)
**Goal:** flash it, feed it the mic, measure real latency and power.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: On the Board</b></summary>

**Timing (~40 min).** 10 min toolchain and flashing · 10 min latency measurement · 10 min RAM high-water · 10 min energy and the closing synthesis.

**Budget setup time honestly, because the toolchain is the actual risk.** The pico-sdk needs a cross-compiler, environment variables, and cmake; SDK versions drift and the flashing path differs across operating systems. **Have every student build and flash the SDK's blink example before the session**, as homework, and bring pre-flashed spares. A room where twelve people are debugging cmake is a room where nobody learns TinyML.

**Frame the session's purpose precisely: everything until now was predicted, and this is the measurement.** Session 1 computed a RAM budget and a latency estimate from the datasheet. **Now find out whether the arithmetic was right.** That framing turns the session from a demo into an experiment with a falsifiable prediction, and it is the reason the three measurements below are worth doing rather than just reading about.

**Latency: teach the GPIO-toggle method, because it is the honest one.** Set a pin high before the inference call, low after, and read it on a scope — that measures the real thing with nanosecond resolution and no instrumentation overhead. `time_us_32()` is the cheaper route and is fine, but note it costs a function call inside the measured region. **Compare the measured number against the Session 1 estimate of MACs × cycles-per-MAC ÷ clock**; a factor-of-two discrepancy is normal and worth diagnosing, a factor of ten means something is wrong.

**RAM high-water: the stack-painting trick is genuinely clever and worth doing slowly.** Fill the stack region with a known pattern (0xAA) at startup, run the workload, then scan for the deepest address that no longer holds the pattern. **That is your true peak stack usage**, and it is otherwise essentially unmeasurable. On a device with no MMU and no stack guard, a stack overflow does not crash — it silently corrupts adjacent memory and the model returns nonsense. **This measurement is the difference between "it works" and "it works reliably."**

**Energy: a USB power meter is enough, and the arithmetic is the interesting part.** At roughly 30 mW and tens of milliseconds per inference, one inference costs under a millijoule. A CR2032 coin cell holds about 1000 J. **Have the room compute the duty cycle needed for a month of battery life** — the answer usually surprises people and is the entire commercial case for TinyML. It also reframes the compression workshop: every factor of two in model size was a factor of two in battery life.

**Point at the front end, because it is where students will actually get stuck.** The ADC produces samples that must become a spectrogram *on the device*, in Q15 fixed point, within the same budget — that is [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb) §1 doing real work. **The FFT frequently costs more than the network.** Ask the room to measure both separately before optimising either; the instinct is always to optimise the neural network, and the instinct is often wrong.

**Close by walking the whole curriculum backwards in one minute.** [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb) supplies the front end. [Intro to C](../Intro_Programming/Intro_C.ipynb) supplies the runtime. [FPGA](../Intro_FPGA/Intro_FPGA.ipynb) supplies the fixed-point arithmetic. [Model Compression](../Intro_Mach_Learn/Model_Compression.ipynb) supplies the int8 weights. [Intro to CNN](./Intro_CNN/Intro_CNN.ipynb) supplies the architecture. **Six workshops converge on a $6 board running a classifier off a coin cell** — and that convergence, rather than any single technique, is what this session exists to demonstrate.
</details>

```bash
# Raspberry Pi Pico (RP2040) flow:
#   pico-sdk + cmake; main.c = ADC → [Q15 spectrogram: Real_Time_DSP §1] → conv net → LED
cmake -B build && make -C build && cp build/kws.uf2 /media/RPI-RP2/
```

💡 **Intuition.** The measurements that matter on-device: **latency** per inference (toggle a GPIO around the call, read it on a scope — or `time_us_32()`), **RAM high-water** (fill the stack with a pattern, see how much got overwritten), and **energy** per inference (a USB power meter is enough). Expect tens of ms per inference at ~30 mW: a coin cell runs your classifier for weeks. The whole [compression ladder](../Intro_Mach_Learn/Model_Compression.ipynb) exists for this moment.

---
## Where next

- [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb) — the front-end feeding the net.
- [Intro to FPGA](../Intro_FPGA/Intro_FPGA.ipynb) — when the microcontroller runs out of steam.